In [10]:
from exodash.utils.production_sector import ProductionSector
import pandas as pd

eval_files = []
properties = pd.read_csv('/pdo/astronet-data/data/labels/tces-vetting-v01-tois-triageJs-nocentroid-april2025-all.csv')
for sector in [85, 86, 87, 88, 89, 90, 91, 92, 93, 94]:
    production_sector = ProductionSector(sector)
    properties = pd.concat([properties, production_sector.properties_df], ignore_index=True)

properties = properties.drop_duplicates(subset=['tic_id', 'astro_id'])
print(properties.columns)
print(properties.isnull().sum().sort_values(ascending=False))





Index(['astro_id', 'dec', 'decision', 'depth', 'duration', 'epoch', 'file',
       'final', 'period', 'ra', 's_mass', 's_rad', 's_rad_est', 'tic_id',
       'tmag', 'as', 'ch', 'disp_b', 'disp_e', 'disp_j', 'disp_n', 'disp_p',
       'disp_t', 'disp_u', 'dm', 'et', 'md', 'mk', 'total_votes',
       'selected_total_votes', 'first_letter', 'planetno', 'centroid_distance',
       'true_label', 'sector', 'unnamed_0', 'centroid_distance_arc_sec'],
      dtype='object')
dm                           44045
centroid_distance_arc_sec    42981
unnamed_0                    42501
decision                     42289
et                           39507
md                           38209
as                           37956
mk                           36761
centroid_distance            36284
ch                           36175
s_rad_est                    35988
selected_total_votes         30039
total_votes                  30039
disp_j                       30039
first_letter                 30039
disp_u

In [8]:
print("\nDuplicate TIC IDs:", properties['tic_id'].duplicated().sum())
print("Duplicate ASTRO IDs:", properties['astro_id'].duplicated().sum())

# If you want to see the *actual rows* that are duplicates:
print("\nRows with duplicate TIC IDs:")
print(properties[properties['tic_id'].duplicated(keep=False)])

print("\nRows with duplicate ASTRO IDs:")
print(properties[properties['astro_id'].duplicated(keep=False)])


Duplicate TIC IDs: 6824
Duplicate ASTRO IDs: 0

Rows with duplicate TIC IDs:
           astro_id        dec decision          depth  duration        epoch  \
1      2.000000e+00  32.167913       eb  350944.764800  0.151577  1792.638064   
16     1.700000e+01  60.580969      NaN   74162.251180  0.128429  1794.741188   
25     2.600000e+01 -32.858269       et   49733.800130  0.130920  1657.834257   
26     2.700000e+01 -30.940627       eb   10608.352470  0.171753  1659.400756   
32     3.300000e+01 -32.567452      NaN    2940.092146  0.069358  1657.286158   
...             ...        ...      ...            ...       ...          ...   
50593  2.026293e+11        NaN      NaN   14628.727581  0.178770  2048.923864   
50594  2.026293e+11        NaN      NaN    4688.690854  0.306523  2061.417079   
50597  2.026637e+11        NaN      NaN   51810.904279  0.284450  2061.249576   
50598  2.026637e+11        NaN      NaN   26551.698254  0.175211  2048.299523   
50599  2.026637e+11        NaN 

In [ ]:
import pandas as pd
import numpy as np

def create_train_spec(
    cluster_results_df,
    default_weight=1.0,
    default_up=1,
    pseudo_weight_fn=None,
    upsample_fn=None,
    output_path="train_spec.parquet"
):
    """
    Create a sparse train_spec parquet file.

    Parameters
    ----------
    cluster_results_df : pd.DataFrame
        Must contain:
            - astro_id
            - tic_id (optional but recommended)
            - cluster_id
            - confidence (optional: your soft pseudo label score)
        
    default_weight : float
        Weight applied to all examples not in cluster_results_df.
    
    default_up : int
        Upsample factor for all examples not in cluster_results_df.

    pseudo_weight_fn : callable
        Function that maps a confidence score to a sample_weight.
        Example: lambda c: np.clip(c, 0.4, 1.0)

    upsample_fn : callable
        Function that maps a confidence score to an upsample factor.
        Example: lambda c: 1 if c < 0.5 else 2 if c < 0.8 else 3

    output_path : str
        File path to write the parquet.

    Returns
    -------
    pd.DataFrame
        spec DataFrame written to parquet.
    """

    df = cluster_results_df.copy()

    # Make sure astro_id is int
    df["astro_id"] = df["astro_id"].astype("int64")

    # Handle missing TIC IDs
    if "tic_id" not in df.columns:
        df["tic_id"] = -1

    # Default columns if not supplied
    if "cluster_id" not in df.columns:
        df["cluster_id"] = -1

    # If confidence not available, assume all pseudo-label strengths = 0.7
    if "confidence" not in df.columns:
        df["confidence"] = 0.7

    # ---- Sample Weights ----
    if pseudo_weight_fn is None:
        # Default: use confidence directly but clipped
        pseudo_weight_fn = lambda c: float(np.clip(c, 0.5, 1.0))

    df["sample_weight"] = df["confidence"].apply(pseudo_weight_fn).astype("float32")

    # ---- Upsample Factors ----
    if upsample_fn is None:
        # Default rule:
        # c < 0.5 → 1×
        # 0.5–0.8 → 2×
        # >0.8 → 3×
        def default_up_fn(c):
            if c < 0.5:
                return 1
            elif c < 0.8:
                return 2
            else:
                return 3
        upsample_fn = default_up_fn

    df["upsample_factor"] = df["confidence"].apply(upsample_fn).astype("int32")

    # Mark these as pseudo examples
    df["is_pseudo"] = True

    # ---- Final Columns ----
    spec = df[[
        "astro_id",
        "tic_id",
        "sample_weight",
        "upsample_factor",
        "is_pseudo",
        "cluster_id",
        "confidence",
    ]]

    # Save parquet
    spec.to_parquet(output_path, index=False)
    print(f"Saved train spec with {len(spec)} rows → {output_path}")

    return spec


# -----------------------
# Example usage:
# -----------------------
if __name__ == "__main__":
    # Suppose your cluster mining output looks like:
    # astro_id, tic_id, cluster_id, confidence
    example_clusters = pd.DataFrame({
        "astro_id": [
            23399041,
            23491633,
            23826331,
            24004619,
            24371322,
            24422764,
            24491836,
        ],
        "tic_id":   [210, 211, 212, 213, 214, 215],
    })

    create_train_spec(example_clusters, output_path="/pdo/users/dimond/train_spec.parquet")